# BQuant Export — Earnings Dates & Ex-Dividends

Produces two CSVs for the IBKR study:

| File | Columns | Purpose |
|---|---|---|
| `earnings_dates.csv` | `ticker, ann_date` | Exclude earnings-window sessions from the ranking |
| `exdiv.csv` | `ticker, ex_date, amount` | Correct fake overnight gaps in the target |

## Why both matter

**Earnings.** In the IBKR panel, 102 overnight moves exceeded 10% — roughly 8.6% of the ~1,190
earnings events across 149 names over two years. Those land in the decile extremes and no price
feature can anticipate them. Your Bloomberg run flagged only 0.8% of rows as earnings-window when
~4.8% was expected, so that variant was never really tested.

**Dividends.** IB `TRADES` bars are split-adjusted but **not** dividend-adjusted. Every ex-date
shows up as a fake overnight gap equal to the dividend — typically 0.3–0.8%, which is small in
absolute terms but *large* relative to the 6 bp edge being measured, and partly predictable from
price features. That is the dangerous kind of contamination: it manufactures signal.

## How to run

Run all cells. Both use a probe-and-report pattern — several BQL field names are attempted and the
notebook prints what each returned, so a field-name change is visible rather than silent. Download
the two CSVs from the BQuant file browser and copy them to `~/ib_work/` on the Mac.

Universe is the current S&P 500. That is a superset of the 149 IBKR tickers; extra rows are
harmless because the merge is on `ticker` + date.

In [ ]:
# ============================ Cell 1 — Setup
import pandas as pd
import numpy as np
import bql

bq = bql.Service()
members = bq.univ.members("SPX Index")

START = "2024-08-01"
END   = "2026-08-22"

print("BQL service OK")
print(f"window: {START} -> {END}")


def raw_frame(field_name, field_obj, universe=None):
    # Parse a BQL response directly. Deliberately does NOT de-duplicate:
    # earnings and dividend histories have MANY rows per ticker, and a
    # drop_duplicates('ticker') would silently collapse them to one.
    resp = bq.execute(bql.Request(universe if universe is not None else members,
                                  {field_name: field_obj}))
    raw = list(resp)[0].df().reset_index()
    raw.columns = [str(c).strip().lower() for c in raw.columns]
    raw = raw.rename(columns={"id": "ticker"})
    raw["ticker"] = raw["ticker"].astype(str).str.split().str[0]
    return raw


def find_date_col(df):
    # Return the first genuine datetime column. Numerics are skipped because
    # pd.to_datetime happily turns floats into 1970 epoch dates.
    for c in df.columns:
        if c == "ticker" or pd.api.types.is_numeric_dtype(df[c]):
            continue
        s = pd.to_datetime(df[c], errors="coerce")
        if s.notna().mean() > 0.5 and s.dropna().dt.year.between(1990, 2100).mean() > 0.9:
            return c, s
    return None, None


def find_amount_col(df, exclude=()):
    for c in df.columns:
        if c in ("ticker",) or c in exclude:
            continue
        v = pd.to_numeric(df[c], errors="coerce")
        if v.notna().mean() > 0.5 and v.dropna().between(0.001, 100).mean() > 0.8:
            return c, v
    return None, None

In [ ]:
# ============================ Cell 2 — Earnings announcement dates
print("=" * 70)
print("EARNINGS ANNOUNCEMENT DATES")
print("=" * 70)

earnings = None

attempts = [
    ("earn_ann_dt_time_hist_with_eps",
     lambda: bq.data.earn_ann_dt_time_hist_with_eps()),
    ("announcement_dt_quarterly",
     lambda: bq.data.announcement_dt(fa_period_type="Q",
                                     fa_period_offset=bq.func.range(-12, 1))),
    ("expected_report_dt", lambda: bq.data.expected_report_dt()),
    ("latest_announcement_dt", lambda: bq.data.latest_announcement_dt()),
]

for name, builder in attempts:
    try:
        raw = raw_frame(name, builder())
    except Exception as e:
        print(f"{name:<32} FAILED  {type(e).__name__}: {e}")
        continue

    print(f"{name:<32} {len(raw):>7,} rows, columns {raw.columns.tolist()}")
    col, parsed = find_date_col(raw)
    if col is None:
        print(f"{'':<32} -> no usable date column")
        continue

    got = pd.DataFrame({"ticker": raw["ticker"],
                        "ann_date": parsed.dt.tz_localize(None).dt.normalize()}).dropna()
    got = got.drop_duplicates()
    in_window = got[(got["ann_date"] >= START) & (got["ann_date"] <= END)]
    print(f"{'':<32} -> date col '{col}', {len(got):,} dates, "
          f"{got['ticker'].nunique():,} tickers, "
          f"{len(in_window):,} inside window "
          f"({got['ann_date'].min().date()} -> {got['ann_date'].max().date()})")

    if len(in_window) > 500:
        earnings = in_window.sort_values(["ticker", "ann_date"]).reset_index(drop=True)
        print(f"\nUSING -> {name}")
        break

if earnings is None:
    print("\n*** No usable earnings source. Modelling will proceed without the filter. ***")
else:
    earnings.to_csv("earnings_dates.csv", index=False)
    per_ticker = earnings.groupby("ticker").size()
    print(f"\nwrote earnings_dates.csv  ({len(earnings):,} rows)")
    print(f"tickers covered   : {earnings['ticker'].nunique():,}")
    print(f"dates per ticker  : median {per_ticker.median():.0f} "
          f"(expect ~8 for two years of quarterly reporting)")
    if per_ticker.median() < 6:
        print("WARNING: fewer than ~6 per ticker means incomplete history --")
        print("         the earnings filter will under-flag, as it did on Bloomberg.")
    display(earnings.head())

In [ ]:
# ============================ Cell 3 — Ex-dividend dates and amounts
print("=" * 70)
print("EX-DIVIDEND DATES")
print("=" * 70)

dividends = None

dvd_attempts = [
    ("dvd_hist_all", lambda: bq.data.dvd_hist_all()),
    ("dvd_hist", lambda: bq.data.dvd_hist()),
    ("dvd_hist_gross", lambda: bq.data.dvd_hist_gross()),
    ("eqy_dvd_hist_gross", lambda: bq.data.eqy_dvd_hist_gross()),
    ("dvd_ex_dt", lambda: bq.data.dvd_ex_dt()),
]

for name, builder in dvd_attempts:
    try:
        raw = raw_frame(name, builder())
    except Exception as e:
        print(f"{name:<24} FAILED  {type(e).__name__}: {e}")
        continue

    print(f"{name:<24} {len(raw):>7,} rows, columns {raw.columns.tolist()}")

    # prefer an explicitly named ex-date column when one exists
    ex_cols = [c for c in raw.columns if "ex_dt" in c or "ex_date" in c]
    if ex_cols:
        col = ex_cols[0]
        parsed = pd.to_datetime(raw[col], errors="coerce")
    else:
        col, parsed = find_date_col(raw)
    if col is None:
        print(f"{'':<24} -> no usable date column")
        continue

    amt_col, amt = find_amount_col(raw, exclude=(col,))
    if amt_col is None:
        print(f"{'':<24} -> date col '{col}' found but NO amount column; skipping")
        continue

    got = pd.DataFrame({"ticker": raw["ticker"],
                        "ex_date": parsed.dt.tz_localize(None).dt.normalize(),
                        "amount": amt}).dropna()
    got = got[(got["ex_date"] >= START) & (got["ex_date"] <= END)]
    got = got.drop_duplicates()
    print(f"{'':<24} -> date '{col}', amount '{amt_col}', {len(got):,} rows in window")

    if len(got) > 200:
        dividends = got.sort_values(["ticker", "ex_date"]).reset_index(drop=True)
        print(f"\nUSING -> {name}")
        break

if dividends is None:
    print("\n*** No usable dividend source. ***")
    print("Run `build` WITHOUT --exdiv; ex-div dates will remain as small fake gaps.")
else:
    dividends.to_csv("exdiv.csv", index=False)
    per_ticker = dividends.groupby("ticker").size()
    print(f"\nwrote exdiv.csv  ({len(dividends):,} rows)")
    print(f"tickers covered  : {dividends['ticker'].nunique():,}")
    print(f"events per ticker: median {per_ticker.median():.0f} (expect ~8 if quarterly)")
    print(f"amount range     : {dividends['amount'].min():.3f} "
          f"-> {dividends['amount'].max():.3f}")
    if dividends["amount"].max() > 20:
        print("WARNING: implausibly large amounts -- check the column is per-share cash,")
        print("         not a yield or a special-dividend flag.")
    display(dividends.head())

In [ ]:
# ============================ Cell 4 — Summary
print("=" * 70)
print("EXPORT SUMMARY")
print("=" * 70)

import os
for fn in ("earnings_dates.csv", "exdiv.csv"):
    if os.path.exists(fn):
        d = pd.read_csv(fn)
        print(f"{fn:<22} {len(d):>7,} rows   {os.path.getsize(fn)/1024:>7.1f} KB")
        print(f"{'':<22} columns: {d.columns.tolist()}")
    else:
        print(f"{fn:<22} NOT CREATED")

print("\nNext: download both CSVs from the BQuant file browser (left panel),")
print("copy them to ~/ib_work/ on the Mac, then run:")
print("\n    python ib_loader_v2.py build --exdiv exdiv.csv\n")
print("Send me the diagnostic output plus both CSVs and I'll wire the")
print("earnings filter into the modelling notebook.")

In [ ]:
names = sorted(n for n in dir(bq.data) if not n.startswith('_'))
print(f"{len(names)} data items available\n")

def show(label, keys):
    hits = [n for n in names if any(k in n.lower() for k in keys)]
    print(f"--- {label} ({len(hits)}) ---")
    for h in hits:
        print("   ", h)

show("DIVIDEND",  ["dvd", "divid"])
show("EARNINGS",  ["earn", "ann_", "announce", "report"])
show("DATE-ish",  ["_dt", "_date"])

In [ ]:
# Probe on a single ticker. Prints columns + first rows so we can see the shape.
T = "IBM US Equity"
RNG = bq.func.range('2024-08-01', '2026-08-22')

candidates = [
    ("dividends",                 lambda: bq.data.dividends()),
    ("ex_dvd_schedule",           lambda: bq.data.ex_dvd_schedule()),
    ("ex_dvd_calendar",           lambda: bq.data.ex_dvd_calendar()),
    ("dividend_ex_date_rng",      lambda: bq.data.dividend_ex_date(dates=RNG)),
    ("dvd_ex_dt_rng",             lambda: bq.data.dvd_ex_dt(dates=RNG)),
    ("reg_cash_dvd_amt_rng",      lambda: bq.data.regular_cash_dividend_amount(dates=RNG)),
    ("announcement_dt_q",         lambda: bq.data.announcement_dt(
                                      fa_period_type='Q', fa_period_offset=bq.func.range(-8, 1))),
    ("announce_dt_q",             lambda: bq.data.announce_dt(
                                      fa_period_type='Q', fa_period_offset=bq.func.range(-8, 1))),
    ("earnings_conf_call_dt_q",   lambda: bq.data.earnings_conf_call_dt(
                                      fa_period_type='Q', fa_period_offset=bq.func.range(-8, 1))),
    ("last_report_date",          lambda: bq.data.last_report_date()),
]

for name, build in candidates:
    try:
        d = list(bq.execute(bql.Request(T, {name: build()})))[0].df().reset_index()
        d.columns = [str(c).strip().lower() for c in d.columns]
        print(f"\n=== {name}  ({len(d)} rows) ===")
        print("columns:", d.columns.tolist())
        print(d.head(4).to_string())
    except Exception as e:
        print(f"\n=== {name}  FAILED: {type(e).__name__}: {str(e)[:150]}")

In [ ]:
T = "IBM US Equity"

tests = [
    ("dividends + dates",
     lambda: bq.data.dividends(dates=bq.func.range('2024-08-01','2026-08-22'))),
    ("dividends + start/end",
     lambda: bq.data.dividends(start='2024-08-01', end='2026-08-22')),
    ("dividends + ex_date_range",
     lambda: bq.data.dividends(ex_date_range=bq.func.range('2024-08-01','2026-08-22'))),
    ("announcement_dt plain",  lambda: bq.data.announcement_dt()),
    ("announce_dt plain",      lambda: bq.data.announce_dt()),
    ("earnings_conf_call_dt",  lambda: bq.data.earnings_conf_call_dt()),
    ("is_eps quarterly",
     lambda: bq.data.is_eps(fpt='Q', fpo=bq.func.range(-8, 0))),
]

for name, build in tests:
    try:
        d = list(bq.execute(bql.Request(T, {name: build()})))[0].df().reset_index()
        d.columns = [str(c).strip().lower() for c in d.columns]
        print(f"\n=== {name}  ({len(d)} rows) ===")
        print("columns:", d.columns.tolist())
        print(d.head(5).to_string())
    except Exception as e:
        print(f"\n=== {name}  FAILED: {type(e).__name__}: {str(e)[:120]}")

In [ ]:
T = "IBM US Equity"

tests = [
    ("dividends fpt/fpo",
     lambda: bq.data.dividends(fpt='Q', fpo=bq.func.range(-8, 0))),
    ("dividends per_share",
     lambda: bq.data.dividends(dvd_typ='REGULAR CASH',
                               dates=bq.func.range('2024-08-01','2026-08-22'))),
    ("is_regular_cash_dividend_per_sh",
     lambda: bq.data.is_regular_cash_dividend_per_sh(fpt='Q', fpo=bq.func.range(-8, 0))),
    ("dividend_per_share_12m",
     lambda: bq.data.dividend_per_share_12m(fpt='Q', fpo=bq.func.range(-8, 0))),
]

for name, build in tests:
    try:
        d = list(bq.execute(bql.Request(T, {name: build()})))[0].df().reset_index()
        d.columns = [str(c).strip().lower() for c in d.columns]
        print(f"\n=== {name}  ({len(d)} rows) ===")
        print("columns:", d.columns.tolist())
        print(d.head(10).to_string())
    except Exception as e:
        print(f"\n=== {name}  FAILED: {type(e).__name__}: {str(e)[:120]}")

In [ ]:
T = "IBM US Equity"
RNG = bq.func.range('2025-01-01', '2026-08-22')

tests = [
    ("d2d_tot_return_gross",
     lambda: bq.data.day_to_day_tot_return_gross_dvds(dates=RNG)),
    ("tot_return_index_gross",
     lambda: bq.data.tot_return_index_gross_dvds(dates=RNG)),
    ("eqy_dvd_adjust_fact",
     lambda: bq.data.eqy_dvd_adjust_fact(dates=RNG)),
    ("px_last_ca_adj_full",
     lambda: bq.data.px_last(dates=RNG, ca_adj='FULL')),
    ("px_last_ca_adj_price",
     lambda: bq.data.px_last(dates=RNG, ca_adj='PRICE')),
]

for name, build in tests:
    try:
        d = list(bq.execute(bql.Request(T, {name: build()})))[0].df().reset_index()
        d.columns = [str(c).strip().lower() for c in d.columns]
        print(f"\n=== {name}  ({len(d)} rows) ===")
        print("columns:", d.columns.tolist())
        print(d.head(3).to_string())
    except Exception as e:
        print(f"\n=== {name}  FAILED: {str(e)[:200]}")

In [ ]:
T = "IBM US Equity"
RNG = bq.func.range('2025-01-01', '2026-08-22')

r = bq.execute(bql.Request(T, {
    "tr": bq.data.day_to_day_tot_return_gross_dvds(dates=RNG),
    "px": bq.data.px_last(dates=RNG, fill='NA'),
}))
items = list(r)
tr = items[0].df().reset_index(); tr.columns = [c.lower() for c in tr.columns]
px = items[1].df().reset_index(); px.columns = [c.lower() for c in px.columns]

tr = tr[['id','date','tr']].rename(columns={'id':'ticker'})
px = px[['id','date','px']].rename(columns={'id':'ticker'})
d = tr.merge(px, on=['ticker','date']).sort_values('date')
d['date'] = pd.to_datetime(d['date'])
d = d.dropna(subset=['px'])

d['px_ret']   = d['px'].pct_change()
d['excess']   = d['tr'] - d['px_ret']          # tr is a fraction, not a percent
d['prev_px']  = d['px'].shift(1)
d['implied']  = d['excess'] * d['prev_px']

hits = d[d['excess'] > 0.0005]                  # >5bp separates dividends from rounding
print(f"candidate ex-dates: {len(hits)}  (expect ~7 over 20 months)")
print(hits[['date','px','px_ret','tr','excess','implied']].to_string(index=False))